[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Selecting Columns


## What you will be able to do

Pick out one column or several from a table by name, tell a Series from a table of one column, and
choose the way of writing a selection that keeps working when a column's name is awkward.

## The idea

### The problem

Most questions about a table need only part of it. A station's mean for the year needs the `station`
and `celsius` columns, and the times can stay out of it. A table read from a real export often has
dozens of columns: identifiers, flags, units, notes, the same value in two units. Before any
calculation, the columns that matter have to be picked out, and the rest left behind.

pandas offers two ways of writing that for a single column, `readings["celsius"]` and
`readings.celsius`. The second is shorter, appears in a great deal of code you will read, and fails
for some column names. A column called `air temp` has a space in its name, so the dot cannot reach
it. A column called `count` shares its name with a method of every DataFrame, so the dot returns the
method. Assigning to `readings.fahrenheit` looks like making a new column and sets an attribute
instead. Python stops at the space with a `SyntaxError`, but the other two mistakes run, with at
most a warning.

### What selecting a column is

> In pandas, **selecting columns** means indexing a DataFrame by column label. A single label in square brackets,
> `df["celsius"]`, returns that column as a **Series**. A list of labels, `df[["station", "celsius"]]`,
> returns a new **DataFrame** holding those columns in the order the list gives. Attribute access,
> `df.celsius`, is a shortcut for the single label that works only for a name that is a valid Python
> identifier and is not already the name of a method or attribute of a DataFrame.

### Why it works that way

A DataFrame keeps its column labels in an index of their own, `df.columns`, as it keeps its row labels
in `df.index`. Square brackets look a label up in `df.columns`, so any label works there, spaces and
punctuation included, and a missing one raises `KeyError`.

The dot goes through Python's attribute lookup instead. Python checks the DataFrame's own methods and
attributes first, `count`, `mean`, `max`, `shape`, `T` and many more, and only then pandas
looks for a column of that name. A column whose name collides with one of them can never be
reached with the dot. Assigning with the dot to a name that is not yet a column sets an ordinary
Python attribute on the object, and pandas warns that no column was made.

The difference between one label and a list of one label also matters. `df["celsius"]` is a Series
with a name, and `df[["celsius"]]` is a DataFrame with one column. Some methods return different
shapes for the two, and functions that expect a table, such as a scikit-learn model's `fit`, want the
second.

### Where this shows up

Selecting columns comes into nearly every line of pandas code. scikit-learn's models take their
features as a table, so the **scikit-learn, Deep Dive** guide will select them with a list of names. The
**Inspecting** notebook listed the columns with `df.columns`, and the **loc and iloc** notebook selects
rows and columns together.

### What this notebook covers

- one column with brackets and with the dot, and a list of columns
- a Series against a table of one column
- a name with a space in it
- columns chosen by type, dropped, renamed and put in a new order
- four errors, among them a column the dot cannot reach and one it did not make

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import pandas as pd

readings = pd.DataFrame({
    "station": ["Bergen", "Oslo"],
    "time": ["00:00", "00:00"],
    "celsius": [8.2, 6.1],
})
print(readings["celsius"])
print(readings[["station", "celsius"]])
```

```
0    8.2
1    6.1
Name: celsius, dtype: float64
  station  celsius
0  Bergen      8.2
1    Oslo      6.1
```

One name in brackets returned the column as a Series, with the column's name as the Series' name. A
list of two names returned a smaller table with those columns and without `time`. Both kept the
table's index, 0 and 1.

## Setup

One library, three helpers for building the readings, and two tables.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings

`year_of_readings` makes a reading for every hour of 2025 at the four stations in `STATIONS`, by the
same formula as the **sqlite3, Deep Dive** guide, with Svalbard's 24 readings for 2 March missing.
`readings` is the table of all 35,040, with the columns `station`, `time` and `celsius`, and `stations`
holds each station's latitude, Kirkenes included. The Common errors section uses `warnings`, and
imports it there.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### A single column with brackets

In [2]:
celsius = readings["celsius"]

print(type(celsius).__name__, celsius.name, celsius.shape)
celsius.head()

Series celsius (35040,)


0    -3.6
1    -3.5
2   -14.6
3    -6.7
4    -3.7
Name: celsius, dtype: float64

The column comes back as a Series named `celsius`, with the table's index. Its shape is `(35040,)`,
one dimension, because a Series has no columns.

### The same column with the dot

In [3]:
print(readings.celsius.equals(readings["celsius"]))

True


`equals` compares two Series value by value, and `True` says the dot returned the same column. When a
column's name is a valid Python name and no DataFrame method or attribute has it, the two forms read
the same column.

### Several columns with a list

A list of names inside the brackets returns a table:

In [4]:
readings[["station", "celsius"]].head(3)

,station,celsius
0,Bergen,-3.6
1,Oslo,-3.5
2,Svalbard,-14.6


The outer brackets select and the inner ones make the list. The columns come back in the order the
list names them, so the same call can reorder them:

In [5]:
readings[["time", "station", "celsius"]].head(3)

,time,station,celsius
0,2025-01-01,Bergen,-3.6
1,2025-01-01,Oslo,-3.5
2,2025-01-01,Svalbard,-14.6


### A Series against a single-column table

In [6]:
as_series = readings["celsius"]
as_table = readings[["celsius"]]

print(type(as_series).__name__, as_series.shape)
print(type(as_table).__name__, as_table.shape)

Series (35040,)
DataFrame (35040, 1)


One label gives a Series of shape `(35040,)`. A list holding the same label gives a DataFrame of shape
`(35040, 1)`, 35,040 rows and one column. The values are the same, and the shape decides what a
function that receives them can do with them.

### A name with a space

Exports often have names such as `air temp`. `rename` gives a column a new name, and brackets reach it
however it is spelled:

In [7]:
renamed = readings.rename(columns={"celsius": "air temp"})

print(renamed.columns.tolist())
print(renamed["air temp"].max())

['station', 'time', 'air temp']
20.8


`renamed.air temp` is not valid Python at all, because the space ends the name after `air`. With
brackets, the label is a string, and a string can hold anything. `rename` returned a new table, and
`readings` keeps its own names.

### Columns chosen by type

`select_dtypes` keeps the columns of the types you name:

In [8]:
print(readings.select_dtypes("number").columns.tolist())
print(readings.select_dtypes(include=["object", "datetime"]).columns.tolist())

['celsius']
['station', 'time']


`"number"` matches every numeric type, so the readings' `float64` column is the one it keeps. The
second call keeps the text and the dates. On a table with dozens of columns, this picks out every
measurement without listing them by name.

### Everything but some columns

`drop` with `columns=` returns the table without the columns you name:

In [9]:
without_time = readings.drop(columns=["time"])

print(without_time.columns.tolist())
print(readings.columns.tolist())

['station', 'celsius']
['station', 'time', 'celsius']


Like `rename`, `drop` returns a new table, and `readings` still has all three columns. The **Adding
and Changing Columns** notebook comes back to methods that return a new table.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/06-selecting-columns-solutions.ipynb).

**1.** Select the `latitude` column of `stations` with brackets, and print its largest value.

In [10]:
# your code here

**2.** Make a table of `readings` with only the `time` and `celsius` columns, in that order, and show its first three rows.

In [11]:
# your code here

**3.** Print the type and shape of `stations["station"]` and of `stations[["station"]]`.

In [12]:
# your code here

**4.** Rename `celsius` to `temperature (C)` in a new table, and print that column's mean, rounded to two places.

In [13]:
# your code here

**5.** Use `select_dtypes` to make a table of the numeric columns of `stations`, and print its column names.

In [14]:
# your code here

## Common errors

### KeyError: ('station', 'celsius')

In [15]:
readings["station", "celsius"]

KeyError: ('station', 'celsius')

Two names separated by a comma inside one pair of brackets make a tuple, `("station", "celsius")`,
and pandas looked for a single column labeled with that tuple. Put the names in a list, with a second
pair of brackets: `readings[["station", "celsius"]]`.

### No error, and a method where a column was meant

A summary of each station has columns named after the statistics in it:

In [16]:
summary = readings.groupby("station")["celsius"].agg(["count", "mean", "max"]).round(2)
summary

,count,mean,max
station,,,
Bergen,8760,8.00,20.8
Oslo,8760,6.50,19.3
Svalbard,8736,-4.48,8.3
Tromso,8760,3.50,16.2


In [17]:
summary.count

<bound method DataFrame.count of           count  mean   max
station                    
Bergen     8760  8.00  20.8
Oslo       8760  6.50  19.3
Svalbard   8736 -4.48   8.3
Tromso     8760  3.50  16.2>

`count` is a DataFrame method, and Python found the method before pandas could look for a column, so
the cell shows the method with the table it belongs to. `mean` and `max` are methods too. Brackets
reach the column:

In [18]:
summary["count"]

station
Bergen      8760
Oslo        8760
Svalbard    8736
Tromso      8760
Name: count, dtype: int64

### UserWarning: Pandas doesn't allow columns to be created via a new attribute name

The warning's full text adds a link to the documentation, so the cell catches it and prints its
first part:

In [19]:
import warnings

small = readings.head(3).copy()        # three rows of the readings to try things on

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    small.fahrenheit = small["celsius"] * 9 / 5 + 32

for warning in caught:
    print(f"{warning.category.__name__}: {str(warning.message).split(' - see')[0]}")
print(small.columns.tolist())
print(small.fahrenheit.round(2).tolist())

['station', 'time', 'celsius']
[25.52, 25.7, 5.72]


The assignment set a Python attribute called `fahrenheit` on the object, and `small` still has its
three columns. Reading `small.fahrenheit` finds the attribute and returns the values, so code that
uses it runs, and the mistake shows up only when the table is saved or printed without the column. Assign with brackets, which
makes a column: `small["fahrenheit"] = small["celsius"] * 9 / 5 + 32`. The **Adding and Changing
Columns** notebook covers it.

### KeyError: "['Celsius'] not in index"

In [20]:
readings[["station", "Celsius"]]

KeyError: "['Celsius'] not in index"

A list of names raises when any name in it is missing, and the message lists the ones it could not
find. Here the index it means is `readings.columns`, the column labels, and the name there is
`celsius`, with a small c, as `readings.columns.tolist()` lists it.

## Recap

- Select one column as a Series with `df["name"]`, and several as a table with a list of names
- Use the dot only for simple names that no DataFrame method shares
- Tell a Series of shape `(n,)` from a table of one column, shape `(n, 1)`
- Choose columns by type with `select_dtypes`, leave some out with `drop`, and rename with `rename`
- Make new columns with brackets, because the dot only sets an attribute

## What is next

The **loc and iloc** notebook selects rows as well as columns, by label with `loc` and by position
with `iloc`, and shows where the two part ways.

---

&#8592; **Previous:** [Inspecting](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/05-inspecting.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [loc and iloc](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/07-loc-and-iloc.ipynb) &#8594;
